In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/RUL_FD002.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/test_FD003.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/Damage Propagation Modeling.pdf
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/readme.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/train_FD003.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/test_FD004.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/train_FD004.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/test_FD002.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/train_FD001.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/train_FD002.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/RUL_FD001.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/RUL_FD004.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/RUL_FD003.txt
/kaggle/input/datasets/suriyachayatummagoon/cmapssdata/test_FD001.txt


In [2]:
from pathlib import Path
import os

print("Looking for FD001 files under /kaggle/input/...")
found = []
for p in sorted(Path("/kaggle/input").rglob("*FD001.txt")):
    print("  ", p)
    found.append(p)

if not found:
    print("\n❌ NO FD001 FILES FOUND — dataset attach nahi hua!")
    print("Right sidebar 'Add Input' -> CMAPSSData pe '+' click karo")
else:
    print(f"\n✅ Found {len(found)} FD001 file(s). Ready to run.")

Looking for FD001 files under /kaggle/input/...
   /kaggle/input/datasets/suriyachayatummagoon/cmapssdata/RUL_FD001.txt
   /kaggle/input/datasets/suriyachayatummagoon/cmapssdata/test_FD001.txt
   /kaggle/input/datasets/suriyachayatummagoon/cmapssdata/train_FD001.txt

✅ Found 3 FD001 file(s). Ready to run.


In [3]:
# ============================================================================
# TrustTwin Phase 1 — Kaggle Edition (All Fixes)
# ============================================================================
import os, sys, json, time, math, random, warnings, hashlib, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================ CONFIG ========================================
SEED          = 42
SEEDS         = [42, 52, 62, 72, 82]
FAST_MODE     = True
NUM_CLIENTS   = 5
PARTICIPATION = 1.0
FL_ROUNDS     = 30 if FAST_MODE else 50
LOCAL_EPOCHS  = 1
BATCH_SIZE    = 64
SEQ_LEN       = 30
LR            = 1e-3
HIDDEN        = 64
LSTM_LAYERS   = 2
DROPOUT       = 0.2

TRUST_GAMMA   = 2.0
TRUST_BETA    = 1.0
TRUST_THRESHOLD = 0.3

POISON_STRENGTH  = 5.0
BACKDOOR_SHIFT   = 3.0
BACKDOOR_TRIGGER = 10.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("=" * 78)
print("TrustTwin Phase 1 — Kaggle Edition")
print("=" * 78)
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("Device :", DEVICE, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
print("Mode   :", "FAST" if FAST_MODE else "PAPER")
print("Seeds  :", [SEED] if FAST_MODE else SEEDS)


# ============================ PATHS =========================================
IN_KAGGLE = os.path.exists("/kaggle/input")
PROJECT = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "TrustTwin"

def find_cmapss_files():
    search_root = Path("/kaggle/input") if IN_KAGGLE else (Path.cwd() / "TrustTwin" / "data")
    if not search_root.exists():
        return None, None, None, None
    for train_p in sorted(search_root.rglob("train_FD001.txt")):
        parent = train_p.parent
        test_p = parent / "test_FD001.txt"
        rul_p  = parent / "RUL_FD001.txt"
        if test_p.exists() and rul_p.exists():
            return parent, train_p, test_p, rul_p
    return None, None, None, None

CMAPSS, TRAIN_PATH, TEST_PATH, RUL_PATH = find_cmapss_files()

if CMAPSS is None and IN_KAGGLE:
    candidates = [
        Path("/kaggle/input/datasets/suriya chayatummagoon/cmapssdata"),
        Path("/kaggle/input/cmapssdata"),
        Path("/kaggle/input/cmapssdata/CMAPSSData"),
    ]
    for c in candidates:
        if c.exists():
            t = c / "train_FD001.txt"
            te = c / "test_FD001.txt"
            r = c / "RUL_FD001.txt"
            if t.exists() and te.exists() and r.exists():
                CMAPSS, TRAIN_PATH, TEST_PATH, RUL_PATH = c, t, te, r
                break

if CMAPSS is None or not TRAIN_PATH.exists():
    print("\n❌ C-MAPSS files not found. Contents under /kaggle/input:")
    if IN_KAGGLE:
        for p in sorted(Path("/kaggle/input").rglob("*.txt"))[:30]:
            print("  ", p)
    raise FileNotFoundError("C-MAPSS FD001 missing. Attach CMAPSSData dataset.")

RESULTS = PROJECT / "results_phase1"
FIGURES = PROJECT / "figures_phase1"
CONFIGS = PROJECT / "configs_phase1"
for p in [RESULTS, FIGURES, CONFIGS]:
    p.mkdir(parents=True, exist_ok=True)

print("Env        :", "Kaggle" if IN_KAGGLE else "Local")
print("Project    :", PROJECT)
print("C-MAPSS dir:", CMAPSS)
print("Train file :", TRAIN_PATH.name, "| exists:", TRAIN_PATH.exists())
print("Test file  :", TEST_PATH.name,  "| exists:", TEST_PATH.exists())
print("RUL file   :", RUL_PATH.name,   "| exists:", RUL_PATH.exists())
print("Results dir:", RESULTS)


# ============================ LOAD C-MAPSS ==================================
COLS = ["unit_id","cycle"] + [f"op_{i}" for i in range(1,4)] + [f"sensor_{i}" for i in range(1,22)]

train = pd.read_csv(TRAIN_PATH, sep=r"\s+", header=None, names=COLS, engine="python")
test  = pd.read_csv(TEST_PATH,  sep=r"\s+", header=None, names=COLS, engine="python")
rul   = pd.read_csv(RUL_PATH,   sep=r"\s+", header=None).iloc[:,0].astype(float).to_numpy()

print("Train:", train.shape, "| Test:", test.shape)
print("Engines:", train.unit_id.nunique(), "/", test.unit_id.nunique())
print("NaNs:", int(train.isna().sum().sum()), "/", int(test.isna().sum().sum()))

SENSORS = [c for c in COLS if c.startswith("sensor_") and train[c].nunique() > 1]
print("Usable sensors:", len(SENSORS))

rng = np.random.default_rng(SEED)
engines = np.array(sorted(train.unit_id.unique())); rng.shuffle(engines)
nval = max(1, int(round(0.20 * len(engines))))
VAL_ENGINES   = np.sort(engines[:nval])
TRAIN_ENGINES = np.sort(engines[nval:])
print("Train engines:", len(TRAIN_ENGINES), "| Val engines:", len(VAL_ENGINES))

def add_rul(df):
    z = df.copy()
    z["RUL"] = z.groupby("unit_id")["cycle"].transform("max") - z["cycle"]
    return z
train_lab = add_rul(train)

scaler = StandardScaler().fit(train_lab[train_lab.unit_id.isin(TRAIN_ENGINES)][SENSORS])

def make_sequences(df, ids, seq_len=SEQ_LEN):
    X, y, e = [], [], []
    sub = df[df.unit_id.isin(ids)]
    for eid, g in sub.groupby("unit_id"):
        g = g.sort_values("cycle")
        if len(g) < seq_len: continue
        z = scaler.transform(g[SENSORS]).astype(np.float32)
        t = g["RUL"].to_numpy(np.float32)
        for i in range(seq_len - 1, len(g)):
            X.append(z[i-seq_len+1:i+1]); y.append(t[i]); e.append(int(eid))
    return np.asarray(X, np.float32), np.asarray(y, np.float32), np.asarray(e)

X_train, y_train, _ = make_sequences(train_lab, TRAIN_ENGINES)
X_val,   y_val,   _ = make_sequences(train_lab, VAL_ENGINES)

Xt = []
for eid, g in test.groupby("unit_id"):
    g = g.sort_values("cycle")
    if len(g) < SEQ_LEN:
        raise RuntimeError(f"Test engine {eid} too short")
    z = scaler.transform(g[SENSORS]).astype(np.float32)
    Xt.append(z[-SEQ_LEN:])
X_test = np.stack(Xt).astype(np.float32)
y_test = rul.astype(np.float32)

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)


# ============================ MODEL & UTILS =================================
class SeqDS(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i] if self.y is None else (self.X[i], self.y[i])

class LSTMReg(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.lstm = nn.LSTM(d, HIDDEN, LSTM_LAYERS, batch_first=True,
                            dropout=DROPOUT if LSTM_LAYERS > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(HIDDEN, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.head(self.lstm(x)[0][:, -1]).squeeze(-1)

def build_model(): return LSTMReg(len(SENSORS)).to(DEVICE)
def state_cpu(m): return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
def load_state(m, s): m.load_state_dict({k: v.to(DEVICE) for k, v in s.items()})

def reg_metrics(y, p):
    return {"RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAE":  float(mean_absolute_error(y, p)),
            "R2":   float(r2_score(y, p))}

def eval_reg(m, X, y):
    m.eval()
    with torch.no_grad():
        p = m(torch.tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    return reg_metrics(y, p), p

def train_local(m, X, y, epochs=1, lr=LR, prox_state=None, mu=0.0):
    dl = DataLoader(SeqDS(X, y), batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    lossfn = nn.MSELoss()
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossfn(m(xb), yb)
            if prox_state is not None and mu > 0:
                prox = 0.0
                for n, p in m.named_parameters():
                    prox = prox + ((p - prox_state[n].to(DEVICE)) ** 2).sum()
                loss = loss + 0.5 * mu * prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
            opt.step()
    return m

EFFECTIVE_EPOCHS = FL_ROUNDS * LOCAL_EPOCHS
seed_everything(SEED)
t0 = time.perf_counter()
central = build_model()
central = train_local(central, X_train, y_train, epochs=EFFECTIVE_EPOCHS, lr=LR)
central_time = time.perf_counter() - t0
central_met, _ = eval_reg(central, X_test, y_test)
print(f"Centralized ({EFFECTIVE_EPOCHS} ep):",
      {k: round(v, 4) for k, v in central_met.items()},
      f"| time={central_time:.1f}s")


# ============================ PARTITIONING ==================================
# ============================================================================
# CELL 5 (FIXED) — Partitioning with robust empty-client prevention
# ============================================================================

def partition_engines(ids, n, mode="iid", seed=SEED):
    ids = np.asarray(sorted(ids))
    rng = np.random.default_rng(seed)

    # FIX 1: use count() to correctly count rows per engine
    engine_len = train_lab.groupby("unit_id")["cycle"].count()
    valid = np.array([e for e in ids if engine_len[e] >= SEQ_LEN])

    # FIX 2: also require that the engine can produce at least 1 sequence
    # (technically engine_len >= SEQ_LEN already guarantees this)
    if len(valid) < n:
        raise RuntimeError(f"Only {len(valid)} valid engines, need >= {n}")

    if mode == "iid":
        rng.shuffle(valid)
        return [np.asarray(x) for x in np.array_split(valid, n)]

    if mode == "quantity":
        rng.shuffle(valid)
        w = rng.dirichlet(np.ones(n) * 0.7)
        # FIX 3: enforce minimum 2 engines per client
        cuts = np.rint(np.cumsum(w) * len(valid)).astype(int)
        cuts = np.maximum(cuts, np.arange(1, n + 1) * 2)
        cuts[-1] = len(valid)
        parts, s = [], 0
        for c in cuts:
            parts.append(np.sort(valid[s:c])); s = c
        return parts

    if mode == "temporal":
        # FIX 4: round-robin so each client gets engines spread over time
        sorted_valid = np.sort(valid)
        parts = [[] for _ in range(n)]
        for i, e in enumerate(sorted_valid):
            parts[i % n].append(e)
        return [np.sort(np.asarray(p)) for p in parts]

    if mode == "label":
        # FIX 5: round-robin over sorted-by-RUL engines
        means = train_lab.groupby("unit_id")["RUL"].mean().reindex(valid).to_numpy()
        order = valid[np.argsort(means)]
        parts = [[] for _ in range(n)]
        for i, e in enumerate(order):
            parts[i % n].append(e)
        return [np.sort(np.asarray(p)) for p in parts]

    raise ValueError(mode)


def build_clients(mode="iid", n=NUM_CLIENTS, seed=SEED):
    parts = partition_engines(TRAIN_ENGINES, n, mode, seed)
    clients = {}
    for cid, eids in enumerate(parts):
        X, y, _ = make_sequences(train_lab, eids)
        if len(X) == 0:
            raise RuntimeError(f"Client {cid} empty — partition mode={mode}")
        clients[cid] = {"X": X, "y": y, "n": len(X), "engines": eids.tolist()}
    return clients


# ---- Pre-flight check: all partitions must have non-empty clients ----
print("Pre-flight partition check:")
for mode in ["iid", "quantity", "temporal", "label"]:
    try:
        _c = build_clients(mode, NUM_CLIENTS, SEED)
        sizes = [v["n"] for v in _c.values()]
        print(f"  {mode:>10s}: OK, sizes={sizes}, total={sum(sizes)}")
    except Exception as e:
        print(f"  {mode:>10s}: FAIL — {e}")
# ============================ AGGREGATORS ===================================
def _flat(state):
    return torch.cat([state[k].flatten() for k in state if torch.is_floating_point(state[k])])

def _unflat(state, flat):
    out, i = {}, 0
    for k in state:
        if torch.is_floating_point(state[k]):
            n = state[k].numel()
            out[k] = flat[i:i+n].view_as(state[k]); i += n
        else:
            out[k] = state[k].clone()
    return out

def agg_weighted(states, weights):
    tot = float(sum(weights))
    return {k: (sum((w/tot)*s[k] for s, w in zip(states, weights))
                if torch.is_floating_point(states[0][k]) else states[0][k].clone())
            for k in states[0]}

def agg_median(states):
    return {k: (torch.median(torch.stack([s[k] for s in states]), dim=0).values
                if torch.is_floating_point(states[0][k]) else states[0][k].clone())
            for k in states[0]}

def agg_trimmed(states, trim=0.2):
    n = len(states); t = max(1, int(n*trim))
    out = {}
    for k in states[0]:
        if not torch.is_floating_point(states[0][k]):
            out[k] = states[0][k].clone(); continue
        st = torch.stack([s[k] for s in states])
        if n > 2*t:
            st = torch.sort(st, dim=0).values[t:n-t]
        out[k] = st.mean(0)
    return out

def agg_krum(states, f, multi=False):
    n = len(states)
    flats = torch.stack([_flat(s) for s in states])
    D = torch.cdist(flats, flats, p=2)
    k = max(1, n - f - 2)
    scores = []
    for i in range(n):
        ds = torch.sort(D[i])[0]
        scores.append(ds[1:k+1].sum().item())
    scores = np.asarray(scores)
    if multi:
        m = max(1, n - f)
        idx = np.argsort(scores)[:m]
        return agg_weighted([states[i] for i in idx], [1.0]*len(idx))
    return states[int(np.argmin(scores))]

def agg_fltrust(global_state, local_states, ref_X, ref_y):
    ref_model = build_model(); load_state(ref_model, global_state)
    ref_model = train_local(ref_model, ref_X, ref_y, epochs=1)
    ref_upd = {k: ref_model.state_dict()[k].detach().cpu() - global_state[k]
               for k in global_state}
    ref_flat = _flat(ref_upd); ref_norm = ref_flat.norm() + 1e-12
    scores, updates = [], []
    for s in local_states:
        upd = {k: s[k] - global_state[k] for k in global_state}
        uf = _flat(upd)
        sim = float(torch.dot(uf, ref_flat) / (uf.norm()*ref_norm + 1e-12))
        scores.append(max(sim, 0.0)); updates.append(upd)
    if sum(scores) <= 0:
        return agg_weighted(local_states, [1.0]*len(local_states))
    agg = {}
    for k in global_state:
        if torch.is_floating_point(global_state[k]):
            agg[k] = sum(scores[i]*updates[i][k] for i in range(len(updates))) / (sum(scores)+1e-12)
        else:
            agg[k] = global_state[k].clone()
    af = _flat(agg); scale = ref_norm / (af.norm() + 1e-12)
    return _unflat(global_state, af*scale)


# ============================ OBSERVABLE TRUST ==============================
def observable_signals(local_state, global_state, ref_X, ref_y, hist_reliability):
    """Trust signals from OBSERVABLE behavior only. NO ground truth used."""
    upd = {k: local_state[k] - global_state[k] for k in global_state}
    uf = _flat(upd)
    ref_model = build_model(); load_state(ref_model, global_state)
    ref_model = train_local(ref_model, ref_X, ref_y, epochs=1)
    ref_upd = {k: ref_model.state_dict()[k].detach().cpu() - global_state[k]
               for k in global_state}
    rf = _flat(ref_upd)
    cos = float(torch.dot(uf, rf) / (uf.norm()*rf.norm() + 1e-12))
    U = max(cos, 0.0)
    mag_ratio = float(uf.norm() / (rf.norm() + 1e-12))
    Q = 1.0 / (1.0 + abs(math.log(mag_ratio + 1e-12)))
    S = 1.0 / (1.0 + max(0.0, mag_ratio - 2.0))
    R = 1.0 - S
    H = float(hist_reliability)
    return Q, U, H, S, R

def compute_trust(observables, w=(0.2, 0.2, 0.2, 0.2, 0.2)):
    arr = np.asarray(observables, dtype=float)
    raw = w[0]*arr[:,0] + w[1]*arr[:,1] + w[2]*arr[:,2] + w[3]*arr[:,3] - w[4]*arr[:,4]
    if np.ptp(raw) < 1e-12: return np.ones(len(raw))
    return (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

def agg_trust(global_state, local_states, weights, observables,
              gamma=TRUST_GAMMA, beta=TRUST_BETA):
    T = compute_trust(observables)
    risk = np.asarray([o[4] for o in observables])
    a = np.asarray(weights, float) * (np.maximum(T, 0.0) ** gamma) * np.exp(-beta * risk)
    if a.sum() <= 0: a = np.ones_like(a)
    a = a / a.sum()
    out = {}
    for k in global_state:
        if torch.is_floating_point(global_state[k]):
            out[k] = sum(float(a[i])*local_states[i][k] for i in range(len(local_states)))
        else:
            out[k] = global_state[k].clone()
    return out, T, a


# ============================ ATTACK INJECTORS ==============================
def attack_poison(local_state, global_state, strength=POISON_STRENGTH):
    return {k: (global_state[k] + strength*(local_state[k] - global_state[k])
                if torch.is_floating_point(local_state[k]) else local_state[k].clone())
            for k in local_state}

def attack_backdoor(local_state, global_state, shift=BACKDOOR_SHIFT):
    out = {}
    for k in local_state:
        if torch.is_floating_point(local_state[k]):
            out[k] = global_state[k] + shift*(local_state[k] - global_state[k])
            out[k] = out[k] + 0.05 * torch.randn_like(out[k]) * BACKDOOR_TRIGGER
        else:
            out[k] = local_state[k].clone()
    return out

def attack_sign_flip(local_state, global_state):
    return {k: (global_state[k] - (local_state[k] - global_state[k])
                if torch.is_floating_point(local_state[k]) else local_state[k].clone())
            for k in local_state}


# ============================ FL RUNNER =====================================
def bytes_of_state(state):
    return int(sum(v.numel()*v.element_size()
                   for v in state.values() if torch.is_floating_point(v)))

def run_fl(method="fedavg", partition="iid", nclients=NUM_CLIENTS,
           participation=PARTICIPATION, rounds=FL_ROUNDS,
           local_epochs=LOCAL_EPOCHS, seed=SEED,
           malicious_ratio=0.0, attack="none",
           use_trust=True, use_dp=False,
           dp_noise=0.0, dp_clip=1.0):
    seed_everything(seed)
    clients = build_clients(partition, nclients, seed)
    global_state = state_cpu(build_model())
    malicious = set(range(max(0, int(round(nclients * malicious_ratio)))))
    hist, trust_hist = [], []
    reliability = {cid: 1.0 for cid in clients}
    total_bytes = 0
    round_times = []
    fedadam_m = {k: torch.zeros_like(v) for k, v in global_state.items()
                 if torch.is_floating_point(v)}
    fedadam_v = {k: torch.zeros_like(v) for k, v in global_state.items()
                 if torch.is_floating_point(v)}
    for r in range(1, rounds + 1):
        t_round = time.perf_counter()
        rng = np.random.default_rng(seed + r)
        k = max(1, int(math.ceil(participation * nclients)))
        selected = sorted(rng.choice(list(clients), size=k, replace=False).tolist())
        local_states, weights, observables = [], [], []
        ref_idx = np.random.default_rng(seed + r + 999).choice(
            len(X_val), size=min(256, len(X_val)), replace=False)
        ref_X, ref_y = X_val[ref_idx], y_val[ref_idx]
        for cid in selected:
            d = clients[cid]
            m = build_model(); load_state(m, global_state)
            if method == "fedprox":
                prox = {n: p.detach().clone() for n, p in m.named_parameters()}
                m = train_local(m, d["X"], d["y"], epochs=local_epochs,
                                prox_state=prox, mu=0.01)
            else:
                m = train_local(m, d["X"], d["y"], epochs=local_epochs)
            local_state = state_cpu(m)
            if cid in malicious:
                if attack == "poison":
                    local_state = attack_poison(local_state, global_state)
                elif attack == "backdoor":
                    local_state = attack_backdoor(local_state, global_state)
                elif attack == "sign_flip":
                    local_state = attack_sign_flip(local_state, global_state)
            if use_dp and dp_noise > 0:
                upd = {kk: local_state[kk] - global_state[kk] for kk in global_state}
                uf = _flat(upd); nrm = uf.norm()
                scale = min(1.0, dp_clip / (float(nrm) + 1e-12))
                upd = {kk: (v*scale + torch.randn_like(v)*dp_noise*dp_clip
                            if torch.is_floating_point(v) else v.clone())
                       for kk, v in upd.items()}
                local_state = {kk: global_state[kk] + upd[kk] for kk in global_state}
            local_states.append(local_state); weights.append(d["n"])
            total_bytes += bytes_of_state(local_state)
            observables.append(observable_signals(local_state, global_state,
                                                  ref_X, ref_y, reliability[cid]))
        if method == "fedavg":
            global_state = agg_weighted(local_states, weights)
        elif method == "fedprox":
            global_state = agg_weighted(local_states, weights)
        elif method == "median":
            global_state = agg_median(local_states)
        elif method == "trimmed":
            global_state = agg_trimmed(local_states)
        elif method == "krum":
            f = max(1, len(local_states) // 5)
            global_state = agg_krum(local_states, f, multi=False)
        elif method == "multikrum":
            f = max(1, len(local_states) // 5)
            global_state = agg_krum(local_states, f, multi=True)
        elif method == "fltrust":
            global_state = agg_fltrust(global_state, local_states, ref_X, ref_y)
        elif method == "scaffold":
            ref_model = build_model(); load_state(ref_model, global_state)
            ref_model = train_local(ref_model, ref_X, ref_y, epochs=1)
            ref_upd = {kk: ref_model.state_dict()[kk].detach().cpu() - global_state[kk]
                       for kk in global_state}
            base = agg_weighted(local_states, weights)
            global_state = {kk: (base[kk] + 0.05*ref_upd[kk]
                                 if torch.is_floating_point(base[kk]) else base[kk].clone())
                            for kk in base}
        elif method == "fedadam":
            base = agg_weighted(local_states, weights)
            beta1, beta2, tau, lr_srv = 0.9, 0.99, 1e-3, 0.05
            new_state = {}
            for kk in base:
                if torch.is_floating_point(base[kk]):
                    delta = base[kk] - global_state[kk]
                    fedadam_m[kk] = beta1*fedadam_m[kk] + (1-beta1)*delta
                    fedadam_v[kk] = beta2*fedadam_v[kk] + (1-beta2)*(delta**2)
                    m_hat = fedadam_m[kk] / (1 - beta1**r)
                    v_hat = fedadam_v[kk] / (1 - beta2**r)
                    new_state[kk] = global_state[kk] + lr_srv * m_hat / (torch.sqrt(v_hat) + tau)
                else:
                    new_state[kk] = base[kk].clone()
            global_state = new_state
        elif method == "trusttwin":
            if use_trust:
                global_state, T, a = agg_trust(global_state, local_states, weights, observables)
                for i, cid in enumerate(selected):
                    reliability[cid] = 0.9*reliability[cid] + 0.1*float(T[i])
                trust_hist.append({"round": r,
                                   **{f"client_{c}": float(tt)
                                      for c, tt in zip(selected, T)}})
            else:
                global_state = agg_weighted(local_states, weights)
        else:
            raise ValueError(method)
        em = build_model(); load_state(em, global_state)
        vm, _ = eval_reg(em, X_val, y_val)
        hist.append({"round": r, **vm, "round_time_s": time.perf_counter() - t_round})
        round_times.append(time.perf_counter() - t_round)
    fm = build_model(); load_state(fm, global_state)
    tm, _ = eval_reg(fm, X_test, y_test)
    tm.update({
        "comm_bytes": total_bytes,
        "avg_round_time_s": float(np.mean(round_times)),
        "total_time_s": float(np.sum(round_times)),
        "trust_final_mean": float(np.mean(list(reliability.values()))),
    })
    return global_state, pd.DataFrame(hist), pd.DataFrame(trust_hist), tm

print("Sanity: TrustTwin IID 5 rounds...")
_, h_s, th_s, tm_s = run_fl(method="trusttwin", partition="iid", rounds=5, seed=SEED)
print("Sanity:", {k: round(v, 4) if isinstance(v, float) else v for k, v in tm_s.items()})


# ============================ 10-METHOD MULTI-SEED ==========================
METHODS = ["fedavg", "fedprox", "scaffold", "fedadam",
           "median", "trimmed", "krum", "multikrum", "fltrust", "trusttwin"]
SEEDS_TO_RUN = [SEED] if FAST_MODE else SEEDS

rows = []
for seed in SEEDS_TO_RUN:
    print(f"\n--- Seed {seed} ---")
    for method in METHODS:
        t0 = time.perf_counter()
        _, h, th, tm = run_fl(method=method, partition="iid", seed=seed)
        rows.append({"Method": method, "Seed": seed, "Partition": "iid",
                     "Attack": "clean", **tm})
        print(f"  {method:>10s}  RMSE={tm['RMSE']:.4f}  MAE={tm['MAE']:.4f}  "
              f"R2={tm['R2']:.4f}  comm={tm['comm_bytes']//1024}KB  "
              f"t={time.perf_counter()-t0:.0f}s")

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv(RESULTS / "fl_baseline_multiseed.csv", index=False)
print("\nSaved:", RESULTS / "fl_baseline_multiseed.csv")
print(baseline_df.groupby("Method")[["RMSE","MAE","R2"]].mean().round(4))

# ============================ STATISTICS ====================================
def mean_std_ci(df, group_col="Method", metric="RMSE"):
    out = []
    for k, g in df.groupby(group_col):
        v = g[metric].dropna().to_numpy(float)
        row = {"Method": k, "N": len(v), "mean": v.mean(),
               "std": v.std(ddof=1) if len(v) > 1 else np.nan}
        if len(v) > 1:
            lo, hi = stats.t.interval(0.95, len(v)-1, loc=v.mean(), scale=stats.sem(v))
            row["ci95_low"], row["ci95_high"] = lo, hi
        out.append(row)
    return pd.DataFrame(out)

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    pooled = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1)) / (n1+n2-2))
    return float((a.mean() - b.mean()) / (pooled + 1e-12))

def wilcoxon_vs_ref(df, ref="fedavg", metric="RMSE"):
    out = []
    ref_v = df[df.Method == ref].groupby("Seed")[metric].mean()
    for m, g in df.groupby("Method"):
        if m == ref:
            continue
        mv = g.groupby("Seed")[metric].mean()
        common = ref_v.index.intersection(mv.index)
        if len(common) >= 3:
            try:
                s, p = stats.wilcoxon(ref_v.loc[common], mv.loc[common])
                out.append({"Method": m, "vs": ref, "n": len(common),
                            "stat": float(s), "p": float(p),
                            "cohens_d": cohens_d(ref_v.loc[common].values,
                                                 mv.loc[common].values)})
            except Exception as e:
                out.append({"Method": m, "vs": ref, "n": len(common), "err": str(e)})
    return pd.DataFrame(out)

stats_df = mean_std_ci(baseline_df, metric="RMSE")
stats_df.to_csv(RESULTS / "stats_rmse.csv", index=False)
print("\nRMSE statistics:")
print(stats_df.round(4))

if len(SEEDS_TO_RUN) >= 3:
    wil = wilcoxon_vs_ref(baseline_df, ref="fedavg", metric="RMSE")
    wil.to_csv(RESULTS / "wilcoxon_vs_fedavg.csv", index=False)
    print("\nWilcoxon vs FedAvg:")
    print(wil.round(4))


# ============================ POISONING SWEEP ===============================
POISON_RATIOS = [0.0, 0.2, 0.4]
POISON_METHODS = ["fedavg", "median", "trimmed", "krum", "multikrum", "trusttwin"]
poison_rows = []
for ratio in POISON_RATIOS:
    for method in POISON_METHODS:
        for seed in SEEDS_TO_RUN:
            _, _, _, tm = run_fl(method=method, partition="iid", seed=seed,
                                 malicious_ratio=ratio,
                                 attack="poison" if ratio > 0 else "none")
            poison_rows.append({"Attack": "poisoning", "Ratio": ratio,
                                "Method": method, "Seed": seed, **tm})
        print(f"  poison ratio={ratio} method={method} done")
poison_df = pd.DataFrame(poison_rows)
poison_df.to_csv(RESULTS / "poison_sweep.csv", index=False)
print("\nPoison sweep (RMSE):")
print(poison_df.groupby(["Ratio","Method"])["RMSE"].mean().unstack().round(4))


# ============================ BACKDOOR + SIGN-FLIP ==========================
ATTACK_METHODS = ["fedavg", "median", "trimmed", "krum", "trusttwin"]
attack_rows = []
for atk in ["backdoor", "sign_flip"]:
    for method in ATTACK_METHODS:
        for seed in SEEDS_TO_RUN:
            _, _, _, tm = run_fl(method=method, partition="iid", seed=seed,
                                 malicious_ratio=0.2, attack=atk)
            attack_rows.append({"Attack": atk, "Method": method,
                                "Seed": seed, **tm})
        print(f"  {atk} method={method} done")
attack_df = pd.DataFrame(attack_rows)
attack_df.to_csv(RESULTS / "attack_sweep.csv", index=False)
print("\nAttack summary (RMSE):")
print(attack_df.groupby(["Attack","Method"])["RMSE"].mean().unstack().round(4))


# ============================ GENUINE ABLATION ==============================
ablation_rows = []
for seed in SEEDS_TO_RUN:
    _, _, _, tm = run_fl(method="trusttwin", seed=seed,
                         malicious_ratio=0.2, attack="poison",
                         use_trust=True, use_dp=False)
    ablation_rows.append({"Variant": "Full TrustTwin", "Seed": seed, **tm})
    _, _, _, tm = run_fl(method="fedavg", seed=seed,
                         malicious_ratio=0.2, attack="poison")
    ablation_rows.append({"Variant": "-Trust", "Seed": seed, **tm})
    _, _, _, tm = run_fl(method="median", seed=seed,
                         malicious_ratio=0.2, attack="poison")
    ablation_rows.append({"Variant": "-Robust Agg", "Seed": seed, **tm})
    _, _, _, tm = run_fl(method="trusttwin", seed=seed,
                         malicious_ratio=0.2, attack="poison",
                         use_dp=True, dp_noise=0.5)
    ablation_rows.append({"Variant": "+DP (0.5)", "Seed": seed, **tm})
    _, _, _, tm = run_fl(method="trusttwin", seed=seed,
                         malicious_ratio=0.0, attack="none")
    ablation_rows.append({"Variant": "Clean (UB)", "Seed": seed, **tm})
    print(f"  ablation seed={seed} done")
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(RESULTS / "genuine_ablation.csv", index=False)
print("\nAblation (RMSE):")
print(ablation_df.groupby("Variant")[["RMSE","MAE","R2"]].mean().round(4))


# ============================ NON-IID COMPARISON ============================
noniid_rows = []
PARTITIONS = ["iid", "quantity", "temporal", "label"]
NONIID_METHODS = ["fedavg", "trusttwin", "trimmed", "multikrum"]
for part in PARTITIONS:
    for method in NONIID_METHODS:
        for seed in SEEDS_TO_RUN:
            _, _, _, tm = run_fl(method=method, partition=part, seed=seed)
            noniid_rows.append({"Partition": part, "Method": method,
                                "Seed": seed, **tm})
        print(f"  partition={part} method={method} done")
noniid_df = pd.DataFrame(noniid_rows)
noniid_df.to_csv(RESULTS / "noniid_phase1.csv", index=False)
print("\nNon-IID (RMSE):")
print(noniid_df.groupby(["Partition","Method"])["RMSE"].mean().unstack().round(4))


# ============================ FIGURES =======================================
plt.figure(figsize=(8, 4.5))
for method in ["fedavg", "trimmed", "krum", "trusttwin"]:
    _, hist, _, _ = run_fl(method=method, seed=SEED)
    plt.plot(hist["round"], hist["RMSE"], label=method)
plt.xlabel("Round"); plt.ylabel("Val RMSE")
plt.title("Convergence (clean IID)")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "fig_convergence.png", dpi=200); plt.close()

_, hist_t, trust_hist, _ = run_fl(method="trusttwin", seed=SEED,
                                  malicious_ratio=0.4, attack="poison")
if len(trust_hist):
    plt.figure(figsize=(8, 4.5))
    for c in [c for c in trust_hist.columns if c.startswith("client_")]:
        plt.plot(trust_hist["round"], trust_hist[c], marker="o", markersize=3, label=c)
    plt.xlabel("Round"); plt.ylabel("Trust")
    plt.title("Trust evolution (40% poisoned)")
    plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.savefig(FIGURES / "fig_trust_evolution.png", dpi=200); plt.close()

plt.figure(figsize=(8, 4.5))
pv = poison_df[poison_df.Method.isin(["fedavg","median","trimmed","krum","trusttwin"])]
pivot = pv.groupby(["Ratio","Method"])["RMSE"].mean().unstack()
for m in pivot.columns:
    plt.plot(pivot.index, pivot[m], marker="o", label=m)
plt.xlabel("Malicious ratio"); plt.ylabel("Test RMSE")
plt.title("Poisoning robustness")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "fig_poison_robustness.png", dpi=200); plt.close()
print("Figures saved to:", FIGURES)


# ============================ SAVE MANIFEST =================================
manifest = {
    "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "python": sys.version, "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "mode": "FAST" if FAST_MODE else "PAPER",
    "seeds": SEEDS_TO_RUN, "methods": METHODS,
    "rounds": FL_ROUNDS, "clients": NUM_CLIENTS,
    "trust_fix": "observable_signals_only_no_ground_truth_leak",
    "cmapss_train_sha256": hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
    "cmapss_test_sha256":  hashlib.sha256(TEST_PATH.read_bytes()).hexdigest(),
    "cmapss_rul_sha256":   hashlib.sha256(RUL_PATH.read_bytes()).hexdigest(),
}
with open(RESULTS / "manifest_phase1.json", "w") as f:
    json.dump(manifest, f, indent=2)

shutil.make_archive("/kaggle/working/trusttwin_phase1", "zip", PROJECT)

print("\n" + "=" * 78)
print("PHASE 1 COMPLETE")
print("=" * 78)
print("Results:", RESULTS)
for f in sorted(os.listdir(RESULTS)): print("  -", f)
print("Figures:", FIGURES)
for f in sorted(os.listdir(FIGURES)): print("  -", f)
print("\nZip: /kaggle/working/trusttwin_phase1.zip")
print("\nRun 'Save Version' from top-right to persist outputs.")

TrustTwin Phase 1 — Kaggle Edition
Python : 3.12.13
Torch  : 2.10.0+cu128
Device : cuda | CUDA: True
GPU    : Tesla T4
Mode   : FAST
Seeds  : [42]
Env        : Kaggle
Project    : /kaggle/working
C-MAPSS dir: /kaggle/input/datasets/suriyachayatummagoon/cmapssdata
Train file : train_FD001.txt | exists: True
Test file  : test_FD001.txt | exists: True
RUL file   : RUL_FD001.txt | exists: True
Results dir: /kaggle/working/results_phase1
Train: (20631, 26) | Test: (13096, 26)
Engines: 100 / 100
NaNs: 0 / 0
Usable sensors: 15
Train engines: 80 | Val engines: 20
X_train: (14459, 30, 15) | X_val: (3272, 30, 15) | X_test: (100, 30, 15)
Centralized (30 ep): {'RMSE': 32.587, 'MAE': 21.4184, 'R2': 0.3851} | time=32.4s
Pre-flight partition check:
         iid: OK, sizes=[2736, 2903, 2645, 2821, 3354], total=14459
    quantity: OK, sizes=[451, 3673, 6646, 3478, 211], total=14459
    temporal: OK, sizes=[2635, 3192, 2740, 3121, 2771], total=14459
       label: OK, sizes=[2796, 2842, 2896, 2930, 2995]

In [4]:
print("train_lab exists :", 'train_lab' in dir())
print("run_fl exists    :", 'run_fl' in dir())
print("baseline_df exists:", 'baseline_df' in dir())
print("poison_df exists :", 'poison_df' in dir())
print("RESULTS exists   :", 'RESULTS' in dir())

train_lab exists : True
run_fl exists    : True
baseline_df exists: True
poison_df exists : True
RESULTS exists   : True


In [5]:
# ============================================================
# TRUSTTWIN — FIXED RERUN (non-IID + figures + zip)
# ============================================================

# ---- 1. Safe partition override (round-robin = no empty clients) ----
def partition_engines(ids, n, mode="iid", seed=SEED):
    ids = np.asarray(sorted(ids))
    rng = np.random.default_rng(seed)
    engine_len = train_lab.groupby("unit_id")["cycle"].count()
    valid = np.array([e for e in ids if engine_len[e] >= SEQ_LEN])
    if len(valid) < 2 * n:
        raise RuntimeError(f"Need >= {2*n} valid engines, have {len(valid)}")
    if mode == "iid":
        rng.shuffle(valid)
    elif mode == "quantity":
        rng.shuffle(valid)
    elif mode in ("temporal", "label"):
        if mode == "label":
            means = train_lab.groupby("unit_id")["RUL"].mean().reindex(valid).to_numpy()
            valid = valid[np.argsort(means)]
        else:
            valid = np.sort(valid)
    else:
        raise ValueError(mode)
    parts = [[] for _ in range(n)]
    for i, e in enumerate(valid):
        parts[i % n].append(e)
    return [np.sort(np.asarray(p)) for p in parts]

# Pre-flight check
print("Pre-flight partition check (FIXED):")
for mode in ["iid", "quantity", "temporal", "label"]:
    try:
        _c = build_clients(mode, NUM_CLIENTS, SEED)
        sizes = [v["n"] for v in _c.values()]
        print(f"  {mode:>10s}: OK, sizes={sizes}")
    except Exception as e:
        print(f"  {mode:>10s}: FAIL — {e}")

# ---- 2. Non-IID rerun (with try/except) ----
noniid_rows = []
for part in ["iid", "quantity", "temporal", "label"]:
    for method in ["fedavg", "trusttwin", "trimmed", "multikrum"]:
        for seed in SEEDS_TO_RUN:
            try:
                _, _, _, tm = run_fl(method=method, partition=part, seed=seed)
                noniid_rows.append({"Partition": part, "Method": method,
                                    "Seed": seed, **tm})
                print(f"  {part}/{method} done")
            except Exception as e:
                print(f"  {part}/{method} FAILED: {e}")

noniid_df = pd.DataFrame(noniid_rows)
noniid_df.to_csv(RESULTS / "noniid_phase1.csv", index=False)
print("\nNon-IID (RMSE):")
print(noniid_df.groupby(["Partition","Method"])["RMSE"].mean().unstack().round(4))

# ---- 3. Figures ----
print("\nGenerating figures...")
plt.figure(figsize=(8, 4.5))
for method in ["fedavg", "trimmed", "krum", "trusttwin"]:
    _, hist, _, _ = run_fl(method=method, seed=SEED)
    plt.plot(hist["round"], hist["RMSE"], label=method)
plt.xlabel("Round"); plt.ylabel("Val RMSE")
plt.title("Convergence (clean IID)")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "fig_convergence.png", dpi=200); plt.close()

_, _, trust_hist, _ = run_fl(method="trusttwin", seed=SEED,
                              malicious_ratio=0.4, attack="poison")
if len(trust_hist):
    plt.figure(figsize=(8, 4.5))
    for c in [c for c in trust_hist.columns if c.startswith("client_")]:
        plt.plot(trust_hist["round"], trust_hist[c], marker="o", markersize=3, label=c)
    plt.xlabel("Round"); plt.ylabel("Trust")
    plt.title("Trust evolution (40% poisoned)")
    plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.savefig(FIGURES / "fig_trust_evolution.png", dpi=200); plt.close()

plt.figure(figsize=(8, 4.5))
pv = poison_df[poison_df.Method.isin(["fedavg","median","trimmed","krum","trusttwin"])]
pivot = pv.groupby(["Ratio","Method"])["RMSE"].mean().unstack()
for m in pivot.columns:
    plt.plot(pivot.index, pivot[m], marker="o", label=m)
plt.xlabel("Malicious ratio"); plt.ylabel("Test RMSE")
plt.title("Poisoning robustness")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES / "fig_poison_robustness.png", dpi=200); plt.close()
print("Figures saved:", FIGURES)

# ---- 4. Manifest + Zip ----
manifest = {
    "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "mode": "FAST" if FAST_MODE else "PAPER",
    "seeds": SEEDS_TO_RUN, "methods": METHODS,
    "rounds": FL_ROUNDS, "clients": NUM_CLIENTS,
}
with open(RESULTS / "manifest_phase1.json", "w") as f:
    json.dump(manifest, f, indent=2)

shutil.make_archive("/kaggle/working/trusttwin_phase1", "zip", PROJECT)

print("\n" + "=" * 78)
print("PHASE 1 COMPLETE")
print("=" * 78)
print("Results:", RESULTS)
for f in sorted(os.listdir(RESULTS)): print("  -", f)
print("Figures:", FIGURES)
for f in sorted(os.listdir(FIGURES)): print("  -", f)
print("\nZip: /kaggle/working/trusttwin_phase1.zip")
print("Ab 'Save Version' click karo top-right me.")

Pre-flight partition check (FIXED):
         iid: OK, sizes=[2865, 2953, 3019, 2799, 2823]
    quantity: OK, sizes=[2865, 2953, 3019, 2799, 2823]
    temporal: OK, sizes=[2635, 3192, 2740, 3121, 2771]
       label: OK, sizes=[2796, 2842, 2896, 2930, 2995]
  iid/fedavg done
  iid/trusttwin done
  iid/trimmed done
  iid/multikrum done
  quantity/fedavg done
  quantity/trusttwin done
  quantity/trimmed done
  quantity/multikrum done
  temporal/fedavg done
  temporal/trusttwin done
  temporal/trimmed done
  temporal/multikrum done
  label/fedavg done
  label/trusttwin done
  label/trimmed done
  label/multikrum done

Non-IID (RMSE):
Method      fedavg  multikrum  trimmed  trusttwin
Partition                                        
iid        24.8119    26.4305  27.8797    33.1811
label      30.3995    34.4875  35.0593    32.4203
quantity   24.8119    26.4305  27.8797    33.1811
temporal   36.3065    26.1450  27.1806    24.3225

Generating figures...
Figures saved: /kaggle/working/figures_p